<a href="https://colab.research.google.com/github/Sujiwankit/DADS5001-HW1-Plotly-express-/blob/main/HW1_Plotly_express_Sujiwan_Kitsutum_6810422019.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thailand Tourism Analysis 2025  
อุตสาหกรรมการท่องเที่ยวถือเป็นหนึ่งในเครื่องยนต์หลักของเศรษฐกิจไทย  
แต่คำถามสำคัญคือ “นักท่องเที่ยวของไทยกำลังมาจากไหน เคลื่อนไหวอย่างไร และกำลังเปลี่ยนไปในทิศทางใด?”

โปรเจกต์นี้จึงมุ่งวิเคราะห์ข้อมูลนักท่องเที่ยวต่างชาติปี 2025  
ผ่านมุมมองเชิงข้อมูล เพื่อเปลี่ยนข้อมูลตัวเลขให้เป็น มุมมองที่ช่วยให้เข้าใจและตัดสินใจได้ดีขึ้น”

## คำถาม

1. โครงสร้างตลาดนักท่องเที่ยวของประเทศไทยในปี 2025 เป็นอย่างไร?  
2. นักท่องเที่ยวเดินทางมาไทย “ช่วงไหนมากที่สุด” และแต่ละประเทศมีจังหวะการเดินทางเหมือนกันหรือไม่?
3. ตลาดนักท่องเที่ยวของไทยกำลังเปลี่ยนไปในทิศทางใด?  

## เครื่องมือที่ใช้
- Pandas  
- NumPy  
- DuckDB  
- Plotly Express  
- Plotly Graph Objects  

In [ ]:
# 0) INSTALL LIBRARIES (Run once in Colab)
!pip -q install duckdb plotly pandas numpy

In [ ]:
# 1) IMPORT LIBRARIES
import pandas as pd
import numpy as np
import duckdb

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from google.colab import files

# Data Loading and Cleaning

ในขั้นตอนนี้ เราจะ:
1. อัปโหลดไฟล์ CSV
2. อ่านข้อมูลดิบ
3. ตั้งชื่อคอลัมน์ใหม่ให้ใช้งานได้ง่าย
4. แปลงข้อมูลเป็นตัวเลข
5. แยกข้อมูลระดับประเทศและระดับภูมิภาค
6. สร้างข้อมูลแบบ long format เพื่อใช้ในการวิเคราะห์รายเดือน
7. ลงทะเบียนตารางใน DuckDB เพื่อใช้ SQL query ได้ทันที

In [ ]:
# 1) LOAD DATA FROM GITHUB (NO UPLOAD NEEDED)
url = "https://raw.githubusercontent.com/Sujiwankit/DADS5001-HW1-Plotly-express-/refs/heads/main/International%20Tourist%20Arrivals%20to%20Thailand%20Jan%20-%20Dec%202025.csv"

raw = pd.read_csv(url, header=None)

print("Raw shape:", raw.shape)
display(raw.head(10))

# 2) DEFINE CLEAN COLUMN NAMES
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

clean_cols = (
    ["nationality"]
    + [f"arr_2025_{m}" for m in months] + ["arr_2025_YTD"]
    + [f"arr_2024_{m}" for m in months] + ["arr_2024_YTD"]
    + [f"pct_change_{m}" for m in months] + ["pct_change_YTD"]
)

print("Expected number of columns:", len(clean_cols))
print(clean_cols)

# 3) SLICE ACTUAL DATA AREA
# หมายเหตุ:
# โครงสร้างไฟล์นี้มี header หลายแถว และมีคอลัมน์แทรกก่อนส่วน %CHANGE
# แนวทางนี้อิงจากโครงสร้างไฟล์ที่ใช้ในโจทย์เดิม

df = raw.iloc[5:116, :41].copy()
df = df.drop(columns=[27])   # remove extra / duplicate column
df.columns = clean_cols

print("Cleaned initial shape:", df.shape)
display(df.head())

# 4) CLEAN TEXT VALUES
for c in df.columns:
    df[c] = df[c].astype(str).str.strip()

df = df.replace({
    "nan": np.nan,
    "": np.nan,
    "#DIV/0!": np.nan,
    "-": np.nan
})

display(df.head())

# 5) CONVERT NUMERIC COLUMNS
numeric_cols = [c for c in df.columns if c != "nationality"]

for c in numeric_cols:
    df[c] = (
        df[c]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
    )
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(df.dtypes)
display(df.head())

# 6) REMOVE EMPTY ROWS
df = df[df["nationality"].notna()].copy()
df = df[df["nationality"].astype(str).str.strip() != ""].copy()

print("Rows after removing empty nationality:", len(df))

# 7) CLASSIFY COUNTRY VS REGION
region_labels = {
    "Asia and the Pacific",
    "South-East Asia",
    "North-East Asia",
    "South Asia",
    "Oceania",
    "Europe",
    "Northern Europe",
    "Western Europe",
    "Central/Eastern Europe",
    "Southern/Medit. Europe",
    "America",
    "Middle East",
    "Africa",
    "Grand Total",
    "Other in Asia",
    "Other in Oceania",
    "Other in Europe",
    "Other in America",
    "Other in Middle East",
    "Other in Africa"
}

df["entity_type"] = np.where(
    df["nationality"].isin(region_labels),
    "Region",
    "Country"
)

df_country = df[df["entity_type"] == "Country"].copy()

print("All rows:", len(df))
print("Country rows:", len(df_country))
print("Region rows:", (df["entity_type"] == "Region").sum())

display(df_country.head())

# 8) CREATE LONG FORMAT FOR MONTHLY ANALYSIS
arr_2025_cols = [f"arr_2025_{m}" for m in months]

df_long_2025 = df.melt(
    id_vars=["nationality", "entity_type", "arr_2025_YTD", "arr_2024_YTD", "pct_change_YTD"],
    value_vars=arr_2025_cols,
    var_name="month",
    value_name="arrivals_2025"
)

df_long_2025["month"] = df_long_2025["month"].str.replace("arr_2025_", "", regex=False)
df_long_2025["month_num"] = df_long_2025["month"].map({
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
})

df_long_2025 = df_long_2025.sort_values(["nationality", "month_num"]).reset_index(drop=True)

print("Long table shape:", df_long_2025.shape)
display(df_long_2025.head(12))

# 9) REGISTER TABLES INTO DUCKDB
con = duckdb.connect()

con.register("tourism_wide", df)
con.register("tourism_country", df_country)
con.register("tourism_long_2025", df_long_2025)

print("DuckDB tables registered successfully:")
print("- tourism_wide")
print("- tourism_country")
print("- tourism_long_2025")

# 10) QUICK VALIDATION
display(con.sql("""
    SELECT nationality, arr_2025_YTD, arr_2024_YTD, pct_change_YTD
    FROM tourism_country
    ORDER BY arr_2025_YTD DESC
    LIMIT 10
""").df())

In [ ]:
# 13) PREMIUM DESIGN SYSTEM
COLOR_BG        = "#0B1020"   # ultra dark navy
COLOR_PANEL     = "#11182D"   # panel navy
COLOR_PANEL_2   = "#182039"   # secondary panel
COLOR_GRID      = "rgba(255,255,255,0.08)"
COLOR_TEXT      = "#F8F1E4"   # warm cream
COLOR_MUTED     = "#CBB89D"
COLOR_ORANGE    = "#F2882F"
COLOR_GOLD      = "#DAA520"
COLOR_RED       = "#B3342C"
COLOR_COPPER    = "#D36228"
COLOR_BROWN     = "#593222"
COLOR_GREEN     = "#4E6E58"
COLOR_TEAL      = "#0F766E"
COLOR_BLUE      = "#2C3E50"
COLOR_CREAM     = "#F5E5C0"
COLOR_WINE      = "#8A3324"

MONTH_ORDER = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

px.defaults.template = "plotly_dark"

TITLE_FONT = dict(family="Georgia", size=24, color=COLOR_TEXT)
SUBTITLE_FONT = dict(family="Georgia", size=12, color=COLOR_MUTED)
AXIS_FONT = dict(family="Georgia", size=12, color=COLOR_TEXT)

def add_premium_layout(
    fig,
    title,
    subtitle,
    height=650,
    showlegend=False,
    coloraxis_showscale=False
):
    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b><br><span style='font-size:12px;color:{COLOR_MUTED}'>{subtitle}</span>",
            x=0.5,
            xanchor="center"
        ),
        paper_bgcolor=COLOR_BG,
        plot_bgcolor=COLOR_PANEL,
        font=dict(color=COLOR_TEXT, family="Georgia"),
        height=height,
        showlegend=showlegend,
        coloraxis_showscale=coloraxis_showscale,
        margin=dict(t=110, l=70, r=50, b=70)
    )
    fig.update_xaxes(
        showgrid=True,
        gridcolor=COLOR_GRID,
        zeroline=False,
        title_font=AXIS_FONT,
        tickfont=AXIS_FONT
    )
    fig.update_yaxes(
        showgrid=False,
        zeroline=False,
        title_font=AXIS_FONT,
        tickfont=AXIS_FONT
    )
    return fig

def fmt_int(x):
    return f"{int(round(x)):,}" if pd.notna(x) else "-"

def fmt_pct(x):
    return f"{x:.1f}%" if pd.notna(x) else "-"

def top_bottom_note(df, value_col, label_col="nationality"):
    top_idx = df[value_col].idxmax()
    bottom_idx = df[value_col].idxmin()
    return (
        df.loc[top_idx, label_col], df.loc[top_idx, value_col],
        df.loc[bottom_idx, label_col], df.loc[bottom_idx, value_col]
    )

# PART 1: Basic – Descriptive Analysis

## คำถามที่ 1
**โครงสร้างตลาดนักท่องเที่ยวของประเทศไทยในปี 2025 เป็นอย่างไร?**
เราจะเริ่มจากการสำรวจ “ภาพใหญ่ของตลาด”  
เพื่อทำความเข้าใจว่าใครคือกลุ่มนักท่องเที่ยวหลักของไทย และตลาดมีการกระจุกตัวมากน้อยเพียงใด
ในส่วนนี้เราจะดูภาพรวมของตลาด โดยพิจารณา:
- ประเทศที่มีนักท่องเที่ยวมากที่สุด
- ส่วนแบ่งตลาดของประเทศหลัก
- ภาพรวมระดับภูมิภาค

เป้าหมายคือสร้างความเข้าใจเบื้องต้นว่าไทยพึ่งพาตลาดใดเป็นหลัก

In [ ]:
# Q1.1 TOP 10 COUNTRIES BY YTD ARRIVALS
q1_top10 = con.sql("""
    SELECT nationality, arr_2025_YTD
    FROM tourism_country
    ORDER BY arr_2025_YTD DESC
    LIMIT 10
""").df()

q1_top10_plot = q1_top10.sort_values("arr_2025_YTD")

leader_country = q1_top10.iloc[0]["nationality"]
leader_value = q1_top10.iloc[0]["arr_2025_YTD"]

fig = px.bar(
    q1_top10_plot,
    x="arr_2025_YTD",
    y="nationality",
    orientation="h",
    color="arr_2025_YTD",
    color_continuous_scale=[
        "#593222", "#8A3324", "#B3342C", "#D36228", "#F2882F", "#F5E5C0"
    ],
    text="arr_2025_YTD"
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    marker_line_color="rgba(255,255,255,0.15)",
    marker_line_width=1.2,
    hovertemplate="<b>%{y}</b><br>YTD Arrivals: %{x:,.0f}<extra></extra>"
)

fig = add_premium_layout(
    fig,
    "Top 10 Tourist Arrivals to Thailand (YTD 2025)",
    "A concentrated market structure is visible when a few countries account for a large share of total arrivals.",
    height=680
)

fig.update_xaxes(title="YTD Arrivals")
fig.update_yaxes(title="")

fig.add_annotation(
    x=leader_value,
    y=leader_country,
    text=f"Top market: <b>{leader_country}</b><br>{fmt_int(leader_value)} arrivals",
    showarrow=True,
    arrowhead=2,
    ax=70,
    ay=-10,
    bgcolor=COLOR_PANEL_2,
    bordercolor="rgba(255,255,255,0.15)",
    font=dict(color=COLOR_TEXT, size=12)
)

fig.show()

**กราฟ Q1.1 TOP 10 COUNTRIES BY YTD ARRIVALS**

แสดงประเทศที่ส่งนักท่องเที่ยวเข้ามาประเทศไทยมากที่สุดในปี 2025 (YTD)
โดยเห็นได้ชัดว่า Malaysia และ China เป็นตลาดหลักที่มีจำนวนนักท่องเที่ยวสูงกว่าประเทศอื่นอย่างโดดเด่น

ข้อสังเกตสำคัญ
*   ตลาดนักท่องเที่ยวไทยมีลักษณะ กระจุกตัวในไม่กี่ประเทศ
*   ประเทศใกล้เคียง (เช่น Malaysia) มีบทบาทสำคัญอย่างมาก
*   ช่องว่างระหว่าง Top 2 กับประเทศอื่นค่อนข้างสูง สะท้อนถึงการพึ่งพาตลาดหลัก

ดังนั้น กราฟนี้ชี้ให้เห็นว่า “Malaysia และ China คือแกนหลักของ demand การท่องเที่ยวไทย” และเป็นตลาดที่มีอิทธิพลต่อภาพรวมของอุตสาหกรรมมากที่สุด

In [ ]:
# Q1.2 MARKET SHARE OF TOP 10
share_sum = q1_top10["arr_2025_YTD"].sum()
leader_share = (q1_top10.iloc[0]["arr_2025_YTD"] / share_sum) * 100

fig = px.pie(
    q1_top10,
    values="arr_2025_YTD",
    names="nationality",
    hole=0.52,
    color_discrete_sequence=[
        "#F5E5C0", "#F2882F", "#D36228", "#B3342C", "#8A3324",
        "#593222", "#4E6E58", "#0F766E", "#2C3E50", "#DAA520"
    ]
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hovertemplate="<b>%{label}</b><br>Share: %{percent}<br>Arrivals: %{value:,.0f}<extra></extra>"
)

fig = add_premium_layout(
    fig,
    "Market Share of Top 10 Tourist Markets",
    "The donut chart highlights whether demand is diversified across countries or dominated by a few major source markets.",
    height=720
)

fig.add_annotation(
    x=0.5, y=0.5,
    text=f"<b>Top 10</b><br><span style='font-size:22px'>{fmt_int(share_sum)}</span><br><span style='font-size:11px;color:{COLOR_MUTED}'>total arrivals</span>",
    showarrow=False,
    font=dict(color=COLOR_TEXT)
)

fig.add_annotation(
    x=1.12, y=0.90, xref="paper", yref="paper",
    text=f"Leading market share<br><b>{leader_country}</b>: {leader_share:.1f}%",
    showarrow=False,
    align="left",
    bgcolor=COLOR_PANEL_2,
    bordercolor="rgba(255,255,255,0.15)",
    font=dict(color=COLOR_TEXT, size=12)
)

fig.show()

**กราฟ Q1.2 MARKET SHARE OF TOP 10**

กราฟ Donut Chart นี้แสดงสัดส่วนของนักท่องเที่ยวจาก 10 ประเทศหลักในปี 2025
โดยเห็นได้ว่า Malaysia และ China มีสัดส่วนสูงที่สุด ใกล้เคียงกันที่ประมาณ 22% ต่อประเทศ

ข้อสังเกตสำคัญ
*   ตลาดนักท่องเที่ยวไทยยังคง กระจุกตัวในประเทศหลักไม่กี่ประเทศ
*   Malaysia และ China รวมกันคิดเป็นสัดส่วนเกือบครึ่งหนึ่งของ Top 10
*   ประเทศอื่น ๆ มีสัดส่วนลดหลั่นลงมาอย่างชัดเจน แสดงถึงความแตกต่างของขนาดตลาด

ความหมายเชิงธุรกิจ
*   ไทยยังคงพึ่งพาตลาดหลักในภูมิภาคเอเชียเป็นหลัก
*   หากเกิดการเปลี่ยนแปลงในตลาดเหล่านี้ อาจส่งผลกระทบต่อภาพรวมได้อย่างมีนัยสำคัญ
ดังนั้น กราฟนี้สะท้อนให้เห็นว่า “โครงสร้าง demand ของการท่องเที่ยวไทยยังคงพึ่งพาตลาดหลักอย่างชัดเจน”

In [ ]:
# Q1.3 REGION COMPARISON
q1_region = con.sql("""
    SELECT nationality, arr_2025_YTD
    FROM tourism_wide
    WHERE entity_type = 'Region'
      AND nationality <> 'Grand Total'
    ORDER BY arr_2025_YTD DESC
""").df()

top_region = q1_region.iloc[0]["nationality"]
top_region_val = q1_region.iloc[0]["arr_2025_YTD"]

fig = px.bar(
    q1_region,
    x="nationality",
    y="arr_2025_YTD",
    color="arr_2025_YTD",
    text="arr_2025_YTD",
    color_continuous_scale=[
        "#2C3E50", "#4E6E58", "#0F766E", "#DAA520", "#F2882F", "#F5E5C0"
    ]
)

fig.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    marker_line_color="rgba(255,255,255,0.15)",
    marker_line_width=1.0,
    hovertemplate="<b>%{x}</b><br>YTD Arrivals: %{y:,.0f}<extra></extra>"
)

fig = add_premium_layout(
    fig,
    "Tourist Arrivals by Region (YTD 2025)",
    "Regional composition reflects Thailand’s geographic advantages, travel accessibility, and dependence on nearby demand hubs.",
    height=680
)

fig.update_xaxes(title="", showgrid=False)
fig.update_yaxes(title="YTD Arrivals", showgrid=True, gridcolor=COLOR_GRID)

fig.add_annotation(
    x=top_region,
    y=top_region_val,
    text=f"Leading region<br><b>{top_region}</b>",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-55,
    bgcolor=COLOR_PANEL_2,
    bordercolor="rgba(255,255,255,0.15)",
    font=dict(color=COLOR_TEXT, size=12)
)

fig.show()

**กราฟ Q1.3 REGION COMPARISON**

แสดงภาพรวมจำนวนนักท่องเที่ยวในระดับภูมิภาค (YTD 2025)
โดยเห็นได้ชัดว่า Asia and the Pacific เป็นภูมิภาคหลักที่มีจำนวนนักท่องเที่ยวสูงที่สุดอย่างทิ้งห่างภูมิภาคอื่น

ข้อสังเกตสำคัญ
*   นักท่องเที่ยวส่วนใหญ่มาจาก ภูมิภาคเอเชีย โดยเฉพาะ Asia and the Pacific และ South-East Asia
*   ภูมิภาคยุโรปและอเมริกามีสัดส่วนรองลงมาอย่างชัดเจน
*   ช่องว่างระหว่างเอเชียกับภูมิภาคอื่นสะท้อนถึงการพึ่งพาตลาดระยะใกล้

ความหมายเชิงธุรกิจ
*   ไทยมีข้อได้เปรียบจาก ระยะทาง การเดินทางสะดวก และต้นทุนต่ำ
*   สำหรับนักท่องเที่ยวในเอเชีย
แม้ยุโรปและอเมริกาจะมีปริมาณน้อยกว่า แต่ยังอาจมีความสำคัญในเชิงรายได้ต่อหัว

ดังนั้น กราฟนี้สะท้อนว่า “โครงสร้างการท่องเที่ยวไทยขับเคลื่อนโดยตลาดเอเชียเป็นหลัก” และเป็นฐานสำคัญของ demand ทั้งระบบ

# PART 2: Intermediate – Analytics

## คำถามที่ 2  
**นักท่องเที่ยวเดินทางมาไทย “ช่วงไหนมากที่สุด” และแต่ละประเทศมีจังหวะการเดินทางเหมือนกันหรือไม่?**

ในส่วนนี้เราจะลงลึกในข้อมูลรายเดือน  
เพื่อทำความเข้าใจ “จังหวะของ demand” ว่าเกิดขึ้นเมื่อใด และแตกต่างกันอย่างไรในแต่ละตลาด

โดยจะพิจารณา:
- เดือนที่มีนักท่องเที่ยวสูงสุด (Peak)
- เดือนที่มีนักท่องเที่ยวต่ำสุด (Low)
- ความแตกต่างของ pattern การเดินทางในแต่ละประเทศ

In [ ]:
# Q2.1 MONTHLY TOTAL ARRIVALS
q2_monthly = con.sql("""
    SELECT month, month_num, SUM(arrivals_2025) AS total_arrivals
    FROM tourism_long_2025
    GROUP BY month, month_num
    ORDER BY month_num
""").df()

peak_row = q2_monthly.loc[q2_monthly["total_arrivals"].idxmax()]
low_row = q2_monthly.loc[q2_monthly["total_arrivals"].idxmin()]

fig = px.line(
    q2_monthly,
    x="month",
    y="total_arrivals",
    markers=True,
    text="total_arrivals"
)

fig.update_traces(
    line=dict(color=COLOR_ORANGE, width=4),
    marker=dict(size=10, color=COLOR_GOLD, line=dict(color=COLOR_CREAM, width=1.3)),
    texttemplate="%{text:,.0f}",
    textposition="top center",
    hovertemplate="<b>%{x}</b><br>Total Arrivals: %{y:,.0f}<extra></extra>"
)

fig = add_premium_layout(
    fig,
    "Monthly Tourist Arrivals to Thailand (2025)",
    "The monthly trend reveals seasonality, highlighting peak and soft periods in inbound tourism demand.",
    height=680
)

fig.update_xaxes(categoryorder="array", categoryarray=MONTH_ORDER, title="", showgrid=False)
fig.update_yaxes(title="Total Arrivals", showgrid=True, gridcolor=COLOR_GRID)

fig.add_annotation(
    x=peak_row["month"], y=peak_row["total_arrivals"],
    text=f"Peak month<br><b>{peak_row['month']}</b>: {fmt_int(peak_row['total_arrivals'])}",
    showarrow=True, arrowhead=2, ax=40, ay=-50,
    bgcolor=COLOR_PANEL_2, bordercolor="rgba(255,255,255,0.15)",
    font=dict(color=COLOR_TEXT, size=12)
)

fig.add_annotation(
    x=low_row["month"], y=low_row["total_arrivals"],
    text=f"Lowest month<br><b>{low_row['month']}</b>: {fmt_int(low_row['total_arrivals'])}",
    showarrow=True, arrowhead=2, ax=-40, ay=50,
    bgcolor=COLOR_PANEL_2, bordercolor="rgba(255,255,255,0.15)",
    font=dict(color=COLOR_TEXT, size=12)
)

fig.show()

**กราฟ Q2.1 MONTHLY TOTAL ARRIVALS**

แสดงแนวโน้มจำนวนนักท่องเที่ยวรวมของไทยในแต่ละเดือนตลอดปี 2025  
โดยเห็นได้ชัดว่า **มีรูปแบบ seasonality อย่างชัดเจน**

#### ข้อสังเกตสำคัญ
- **เดือนมกราคม (Jan)** เป็นช่วงที่มีนักท่องเที่ยวสูงสุด (Peak)  
- จำนวนลดลงต่อเนื่องจนถึง **เดือนกันยายน (Sep)** ซึ่งเป็นจุดต่ำสุด (Low)  
- หลังจากนั้นแนวโน้มเริ่มฟื้นตัวและเพิ่มขึ้นอีกครั้งในช่วงปลายปี  

#### การตีความ
- ต้นปีเป็นช่วง **high season** ของการท่องเที่ยวไทย  
- กลางปีถึงต้นไตรมาส 4 เป็นช่วง **low season**  
- ปลายปีเริ่มเข้าสู่ช่วงฟื้นตัวของ demand  

#### ประโยชน์เชิงกลยุทธ์
- ใช้บริหาร capacity ในช่วง peak และ low season  
- วางแผนทำการตลาดเพื่อกระตุ้นช่วงที่ demand อ่อนตัว  
- สนับสนุนการตั้งราคาแบบ **dynamic pricing** ตามฤดูกาล  

ดังนั้น กราฟนี้ยืนยันว่า **“การท่องเที่ยวไทยมีฤดูกาลที่ชัดเจน และ demand ไม่ได้กระจายเท่ากันตลอดปี”**

In [ ]:
# Q2.2 TOP 5 COUNTRIES SEASONALITY
top5 = con.sql("""
    SELECT nationality
    FROM tourism_country
    ORDER BY arr_2025_YTD DESC
    LIMIT 5
""").df()["nationality"].tolist()

q2_top5 = con.sql(f"""
    SELECT nationality, month, month_num, arrivals_2025
    FROM tourism_long_2025
    WHERE nationality IN {tuple(top5)}
    ORDER BY month_num
""").df()

fig = px.line(
    q2_top5,
    x="month",
    y="arrivals_2025",
    color="nationality",
    markers=True,
    color_discrete_sequence=[
        "#F2882F", "#B3342C", "#4E6E58", "#DAA520", "#2C3E50"
    ]
)

fig = add_premium_layout(
    fig,
    "Seasonality of Top 5 Tourist Markets",
    "Different source markets often peak in different months, reflecting holiday patterns, travel behavior, and cultural timing.",
    height=740,
    showlegend=True
)

fig.update_xaxes(categoryorder="array", categoryarray=MONTH_ORDER, title="", showgrid=False)
fig.update_yaxes(title="Monthly Arrivals", showgrid=True, gridcolor=COLOR_GRID)

# annotate max point of top market only to avoid clutter
top_market = top5[0]
top_market_df = q2_top5[q2_top5["nationality"] == top_market]
top_market_peak = top_market_df.loc[top_market_df["arrivals_2025"].idxmax()]

fig.add_annotation(
    x=top_market_peak["month"],
    y=top_market_peak["arrivals_2025"],
    text=f"{top_market} peak<br><b>{top_market_peak['month']}</b>",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-45,
    bgcolor=COLOR_PANEL_2,
    bordercolor="rgba(255,255,255,0.15)",
    font=dict(color=COLOR_TEXT, size=12)
)

fig.show()

**Q2.2 TOP 5 COUNTRIES SEASONALITY**

กราฟนี้แสดงรูปแบบการเดินทางรายเดือนของ 5 ประเทศหลัก  
โดยเห็นได้ว่าแต่ละประเทศมีช่วงเวลา peak แตกต่างกันอย่างชัดเจน

#### ข้อสังเกตสำคัญ
- **China และ Malaysia** มีปริมาณนักท่องเที่ยวสูง และมี peak ในช่วงต้นปี  
- **Malaysia** มีแนวโน้มค่อนข้างสม่ำเสมอ และมีจุดสูงอีกครั้งในช่วงปลายไตรมาส 3  
- **India** มีแนวโน้มเพิ่มขึ้นอย่างต่อเนื่องในช่วงปลายปี  
- **Korea และ Russian Federation** มีปริมาณน้อยกว่า และมี pattern การฟื้นตัวช่วงปลายปี

#### การตีความ
- ตลาดหลักของไทย **ไม่ได้มี seasonality แบบเดียวกันทั้งหมด**  
- พฤติกรรมการเดินทางขึ้นอยู่กับปัจจัยเฉพาะของแต่ละประเทศ เช่น วันหยุด วัฒนธรรม และฤดูกาลท่องเที่ยว  
- Demand ของไทยจึงเป็นการ “รวมกันของหลาย seasonality”

#### ประโยชน์เชิงกลยุทธ์
- สามารถวางแผน **การตลาดเฉพาะประเทศ (market-specific strategy)** ได้  
- ใช้กระจาย demand เพื่อลดผลกระทบจาก low season  
- เพิ่มประสิทธิภาพในการวางแผนเที่ยวบิน โรงแรม และโปรโมชัน

ดังนั้น กราฟนี้สะท้อนว่า **“แต่ละตลาดมีพฤติกรรมการเดินทางต่างกัน และสามารถใช้ความต่างนี้เป็นโอกาสเชิงกลยุทธ์ได้”**

In [ ]:
# Q2.3 HEATMAP SEASONALITY
pivot_top5 = q2_top5.pivot(index="nationality", columns="month", values="arrivals_2025")
pivot_top5 = pivot_top5[MONTH_ORDER]

max_val = np.nanmax(pivot_top5.values)
max_pos = np.where(pivot_top5.values == max_val)
max_y = pivot_top5.index[max_pos[0][0]]
max_x = pivot_top5.columns[max_pos[1][0]]

# สร้างข้อความตัวเลขสำหรับแสดงในแต่ละช่อง
text_values = pivot_top5.applymap(lambda x: f"{x/1000:.0f}K" if pd.notna(x) else "")

fig = go.Figure(data=go.Heatmap(z=pivot_top5.values,x=pivot_top5.columns,y=pivot_top5.index,
    text=text_values.values,texttemplate="%{text}",textfont={"size": 11, "color": "white"},colorscale=[
        [0.00, "#2C3E50"],[0.25, "#4E6E58"],[0.50, "#0F766E"],[0.75, "#F2882F"],[1.00, "#F5E5C0"]],
    hovertemplate="<b>%{y}</b><br>Month: %{x}<br>Arrivals: %{z:,.0f}<extra></extra>",
    colorbar=dict(title="Arrivals")))
china_jan_val = pivot_top5.loc["China", "Jan"]
fig.add_annotation(x="Jan",y="China",text=f"{china_jan_val/1000:.0f}K",showarrow=False,
    font=dict(color="black",size=11,family="Georgia"))
fig = add_premium_layout(
    fig,
    "Seasonality Heatmap of Top 5 Tourist Markets",
    "The heatmap makes monthly concentration patterns immediately visible and helps compare timing across countries at a glance.",
    height=580)
fig.add_annotation(x=max_x, y=max_y,text=f"Highest monthly cell<br><b>{max_y}</b> in <b>{max_x}</b>",
    showarrow=True,arrowhead=2,ax=80,ay=-40,bgcolor=COLOR_PANEL_2,bordercolor="rgba(255,255,255,0.15)",font=dict(color=COLOR_TEXT, size=12)
)

fig.show()

**Q2.3 HEATMAP SEASONALITY **

Heatmap นี้แสดงรูปแบบ seasonality ของ 5 ประเทศหลักในมุมมองที่อ่านได้รวดเร็ว  
โดยสีที่สว่างกว่าจะหมายถึงเดือนที่มีจำนวนนักท่องเที่ยวสูงกว่า

#### ข้อสังเกตสำคัญ
- **China ในเดือนมกราคม** เป็นจุดที่มีค่าสูงที่สุด (Highest monthly cell)  
- **Malaysia** มีปริมาณสูงและค่อนข้างสม่ำเสมอตลอดปี  
- ประเทศอื่น ๆ เช่น **India, Korea และ Russian Federation** มี pattern ที่แตกต่างกันในแต่ละช่วงเวลา  
- หลายประเทศมีช่วง low ในกลางปี และฟื้นตัวในช่วงปลายปี

#### การตีความ
- แต่ละประเทศมี **ช่วงเวลาที่โดดเด่นไม่เหมือนกัน**  
- Seasonality ของไทยเกิดจากการรวมกันของ demand หลายรูปแบบ  
- Heatmap ทำให้เห็น “จังหวะเวลา” ของแต่ละตลาดได้ชัดเจนในภาพเดียว

#### ข้อดีของ Heatmap
- เห็นเดือนที่เด่นของแต่ละประเทศได้ทันที  
- เปรียบเทียบหลายประเทศพร้อมกันได้ง่าย  
- เหมาะสำหรับสรุปภาพรวมให้เข้าใจเร็วในเชิง visual  

ดังนั้น กราฟนี้ช่วยยืนยันว่า **“seasonality ของการท่องเที่ยวไทยมีความหลากหลาย และขึ้นอยู่กับพฤติกรรมของแต่ละตลาด”**

# PART 3: Advanced – Insight

## คำถามที่ 3  
**ตลาดนักท่องเที่ยวของไทยกำลังเปลี่ยนไปในทิศทางใด?**

หลังจากเข้าใจภาพรวมและ seasonality แล้ว  
คำถามสำคัญคือ “อนาคตของตลาดจะไปทางไหน” และไทยควรโฟกัสที่ใคร

ในส่วนนี้เราจะวิเคราะห์เชิงกลยุทธ์ โดยมองทั้ง “ขนาด” และ “การเติบโต” ของตลาด เพื่อหาโอกาสใหม่

โดยจะพิจารณา:
- ประเทศที่เติบโตเร็ว (Growth markets)
- ประเทศที่เริ่มชะลอหรือหดตัว (Declining markets)
- ประเทศที่มีทั้ง “ขนาดใหญ่ + โตสูง” ซึ่งเป็น **Strategic priority** ในอนาคต

In [ ]:
# Q3.1 TOP GROWTH MARKETS
q3_growth = con.sql("""
    SELECT nationality, pct_change_YTD, arr_2025_YTD
    FROM tourism_country
    WHERE pct_change_YTD IS NOT NULL
    ORDER BY pct_change_YTD DESC
    LIMIT 10
""").df()

growth_leader = q3_growth.iloc[0]

fig = px.bar(
    q3_growth.sort_values("pct_change_YTD"),
    x="pct_change_YTD",
    y="nationality",
    orientation="h",
    color="pct_change_YTD",
    color_continuous_scale=[
        "#593222", "#8A3324", "#B3342C", "#F2882F", "#F5E5C0"
    ],
    text="pct_change_YTD"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    marker_line_color="rgba(255,255,255,0.15)",
    marker_line_width=1.1,
    hovertemplate="<b>%{y}</b><br>YoY Growth: %{x:.1f}%<extra></extra>"
)

fig = add_premium_layout(
    fig,
    "Top 10 Growth Markets (YoY % Change)",
    "Fast-growing markets may represent recovery momentum, newly activated demand, or strategic opportunities for future expansion.",
    height=680
)

fig.update_xaxes(title="YoY Growth (%)", showgrid=True, gridcolor=COLOR_GRID)
fig.update_yaxes(title="")

fig.add_annotation(
    x=growth_leader["pct_change_YTD"],
    y=growth_leader["nationality"],
    text=f"Fastest growth<br><b>{growth_leader['nationality']}</b>: {growth_leader['pct_change_YTD']:.1f}%",
    showarrow=True,
    arrowhead=2,
    ax=70,
    ay=-20,
    bgcolor=COLOR_PANEL_2,
    bordercolor="rgba(255,255,255,0.15)",
    font=dict(color=COLOR_TEXT, size=12)
)

fig.show()

Q3.1 TOP GROWTH MARKETS

กราฟนี้แสดงประเทศที่มีอัตราการเติบโตของนักท่องเที่ยวสูงที่สุด (YoY %)  
โดยเห็นได้ว่า **Iraq และ North Korea มีอัตราการเติบโตโดดเด่นสูงกว่าประเทศอื่นอย่างชัดเจน**

#### ข้อสังเกตสำคัญ
- **Iraq** เป็นประเทศที่เติบโตเร็วที่สุด (Fastest growth)  
- กลุ่มประเทศที่ติดอันดับส่วนใหญ่เป็นตลาดที่ยังมีขนาดไม่ใหญ่มาก  
- การเติบโตสูงอาจสะท้อนถึงการฟื้นตัวหรือ demand ใหม่ที่เพิ่งเกิดขึ้น

#### การตีความ
- ตลาดที่มี growth สูงไม่ได้แปลว่าใหญ่เสมอไป แต่สะท้อน “โมเมนตัมของการเติบโต”  
- ประเทศเหล่านี้อาจเป็น **emerging markets** ที่มีศักยภาพในอนาคต  
- ควรพิจารณาควบคู่กับขนาดตลาดก่อนตัดสินใจเชิงกลยุทธ์

#### ความหมายเชิงกลยุทธ์
- ใช้ระบุ “โอกาสใหม่” สำหรับการขยายตลาด  
- เหมาะสำหรับการทดลองทำการตลาด (market testing)  
- หากตลาดใดมีทั้ง **growth สูง + ขนาดใหญ่** จะกลายเป็น **Strategic priority**

ดังนั้น กราฟนี้สะท้อนว่า **“ตลาดที่เติบโตเร็วคือสัญญาณของโอกาสใหม่ แต่ต้องพิจารณาร่วมกับขนาดตลาดเพื่อกำหนดกลยุทธ์ที่เหมาะสม”**

In [ ]:
# Q3.2 DECLINING MARKETS
q3_decline = con.sql("""
    SELECT nationality, pct_change_YTD, arr_2025_YTD
    FROM tourism_country
    WHERE pct_change_YTD IS NOT NULL
    ORDER BY pct_change_YTD ASC
    LIMIT 10
""").df()

decline_leader = q3_decline.iloc[0]

fig = px.bar(
    q3_decline.sort_values("pct_change_YTD"),
    x="pct_change_YTD",
    y="nationality",
    orientation="h",
    color="pct_change_YTD",
    color_continuous_scale=[
        "#F5E5C0", "#F2882F", "#D36228", "#B3342C", "#593222"
    ],
    text="pct_change_YTD"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    marker_line_color="rgba(255,255,255,0.15)",
    marker_line_width=1.1,
    hovertemplate="<b>%{y}</b><br>YoY Growth: %{x:.1f}%<extra></extra>"
)

fig = add_premium_layout(
    fig,
    "Top 10 Declining Markets (YoY % Change)",
    "Declining markets indicate potential exposure, weaker recovery, or changing travel behavior that may require targeted action.",
    height=680
)

fig.update_xaxes(title="YoY Growth (%)", showgrid=True, gridcolor=COLOR_GRID)
fig.update_yaxes(title="")

fig.add_annotation(
    x=decline_leader["pct_change_YTD"],
    y=decline_leader["nationality"],
    text=f"Steepest decline<br><b>{decline_leader['nationality']}</b>: {decline_leader['pct_change_YTD']:.1f}%",
    showarrow=True,
    arrowhead=2,
    ax=-60,
    ay=10,
    bgcolor=COLOR_PANEL_2,
    bordercolor="rgba(255,255,255,0.15)",
    font=dict(color=COLOR_TEXT, size=12)
)

fig.show()

### Q3.2 DECLINING MARKETS

กราฟนี้แสดงประเทศที่มีอัตราการเติบโตของนักท่องเที่ยวติดลบ (YoY %)  
โดยเห็นได้ว่า **Cambodia เป็นตลาดที่หดตัวมากที่สุด ตามด้วย China และ Vietnam**

#### ข้อสังเกตสำคัญ
- **Cambodia** มีการลดลงสูงสุด (Steepest decline)  
- หลายประเทศในเอเชีย เช่น **China, Vietnam, Hong Kong และ Macao** มีแนวโน้มลดลง  
- การหดตัวไม่ได้เกิดเฉพาะประเทศเดียว แต่เกิดในหลายตลาดพร้อมกัน

#### การตีความ
- ตลาดบางประเทศอาจกำลังเผชิญ **การฟื้นตัวที่ช้าลง หรือพฤติกรรมการเดินทางที่เปลี่ยนไป**  
- การลดลงในตลาดขนาดใหญ่ (เช่น China) อาจส่งผลกระทบต่อภาพรวมของไทย  
- สะท้อนถึงความเสี่ยงจากการพึ่งพาตลาดเดิม

#### ความหมายเชิงกลยุทธ์
- ใช้ระบุ “ตลาดเสี่ยง” ที่ต้องเฝ้าระวัง  
- พิจารณาว่าควร **ฟื้นฟู (revive), รักษา (maintain) หรือปรับลดการพึ่งพา (diversify)**  
- สนับสนุนการกระจายตลาดเพื่อลดความเสี่ยงในระยะยาว  

ดังนั้น กราฟนี้ชี้ให้เห็นว่า **“การเติบโตของตลาดไม่ได้เป็นบวกทั้งหมด และบางตลาดสำคัญกำลังชะลอตัวลง”**

In [ ]:
# Q3.3 SIZE VS GROWTH BUBBLE CHART
import numpy as np

q3_bubble = con.sql("""
    SELECT nationality, arr_2025_YTD, pct_change_YTD
    FROM tourism_country
    WHERE arr_2025_YTD IS NOT NULL
      AND pct_change_YTD IS NOT NULL
""").df()

# FIX: SCALE SIZE (แก้ปัญหาวงกลมเล็ก)
q3_bubble["size_scaled"] = q3_bubble["arr_2025_YTD"] / 1_000_000  # แปลงเป็นล้าน

# MEDIAN LINES (แบ่ง Quadrant)
x_med = q3_bubble["arr_2025_YTD"].median()
y_med = q3_bubble["pct_change_YTD"].median()

# FIND STRATEGIC POINT
priority_df = q3_bubble[
    (q3_bubble["arr_2025_YTD"] >= x_med) &
    (q3_bubble["pct_change_YTD"] >= y_med)
].copy()

priority_df["score"] = priority_df["arr_2025_YTD"] * (priority_df["pct_change_YTD"] + 100)

priority_point = (
    priority_df.sort_values("score", ascending=False).iloc[0]
    if len(priority_df) > 0 else q3_bubble.iloc[0]
)

# PLOT
fig = px.scatter(
    q3_bubble,
    x="arr_2025_YTD",
    y="pct_change_YTD",
    size="size_scaled",  # 👈 ใช้ size ที่ scale แล้ว
    color="pct_change_YTD",
    hover_name="nationality",
    color_continuous_scale="Turbo",
    size_max=65  # 👈 ทำให้ bubble ใหญ่ขึ้นชัดเจน
)

fig = add_premium_layout(
    fig,
    "Tourism Market Portfolio: Size vs Growth",
    "This portfolio view separates large and fast-growing markets from mature, emerging, and weaker segments.",
    height=780,
    coloraxis_showscale=True
)

# AXES
fig.update_xaxes(title="YTD Arrivals", showgrid=True, gridcolor=COLOR_GRID)
fig.update_yaxes(title="YoY Growth (%)", showgrid=True, gridcolor=COLOR_GRID)

# QUADRANT LINES
fig.add_vline(x=x_med, line_dash="dot", line_color="rgba(255,255,255,0.35)")
fig.add_hline(y=y_med, line_dash="dot", line_color="rgba(255,255,255,0.35)")

# QUADRANT LABELS
fig.add_annotation(x=x_med*1.65, y=max(q3_bubble["pct_change_YTD"])*0.92,
                   text="Strategic Priority", showarrow=False,
                   font=dict(color=COLOR_CREAM, size=13))

fig.add_annotation(x=x_med*0.45, y=max(q3_bubble["pct_change_YTD"])*0.92,
                   text="Emerging Market", showarrow=False,
                   font=dict(color=COLOR_CREAM, size=13))

fig.add_annotation(x=x_med*1.65, y=min(q3_bubble["pct_change_YTD"])*0.75,
                   text="Mature / Slow Growth", showarrow=False,
                   font=dict(color=COLOR_CREAM, size=13))

fig.add_annotation(x=x_med*0.45, y=min(q3_bubble["pct_change_YTD"])*0.75,
                   text="Monitor", showarrow=False,
                   font=dict(color=COLOR_CREAM, size=13))

# HIGHLIGHT STRATEGIC POINT
fig.add_annotation(
    x=priority_point["arr_2025_YTD"],
    y=priority_point["pct_change_YTD"],
    text=f"Strategic candidate<br><b>{priority_point['nationality']}</b>",
    showarrow=True,
    arrowhead=2,
    ax=90,
    ay=-40,
    bgcolor=COLOR_PANEL_2,
    bordercolor="rgba(255,255,255,0.15)",
    font=dict(color=COLOR_TEXT, size=12)
)
# BONUS: เพิ่มขอบวงกลมให้ดู premium
fig.update_traces(
    marker=dict(
        line=dict(width=1.5, color="rgba(255,255,255,0.3)")
    )
)

fig.show()

**Q3.3 SIZE VS GROWTH BUBBLE CHART**

กราฟ Bubble นี้แสดง “ขนาดตลาด (YTD Arrivals)” และ “อัตราการเติบโต (YoY %)” พร้อมกัน  
โดยเส้นแบ่งช่วยแยกกลุ่มตลาดออกเป็น 4 Quadrants เพื่อใช้วิเคราะห์เชิงกลยุทธ์

#### ข้อสังเกตสำคัญ
- **India** อยู่ในกลุ่มที่มีทั้งขนาดตลาดและการเติบโตสูง (Strategic candidate)  
- ตลาดขนาดใหญ่บางประเทศมีการเติบโตต่ำหรือเป็นลบ สะท้อนการชะลอตัว  
- ตลาดขนาดเล็กหลายประเทศมีการเติบโตสูง แต่ยังไม่ส่งผลต่อภาพรวมมากนัก

#### วิธีตีความ 4 Quadrants
- **High Size + High Growth**  
  ตลาดหลักเชิงกลยุทธ์ ควรเร่งลงทุนและขยายต่อเนื่อง  

- **High Size + Low Growth**  
  ตลาดใหญ่ที่ยังสำคัญ ควรเน้นรักษาฐานและเพิ่มคุณภาพรายได้  

- **Low Size + High Growth**  
  ตลาดเกิดใหม่ที่มีศักยภาพ เหมาะสำหรับการทดลองและขยายในอนาคต  

- **Low Size + Low Growth**  
  ตลาดที่ยังไม่โดดเด่น ควรติดตามมากกว่าลงทุนหนัก  

#### ความหมายเชิงกลยุทธ์
- การตัดสินใจไม่ควรดูแค่ “ขนาด” หรือ “การเติบโต” อย่างใดอย่างหนึ่ง  
- การมองแบบ portfolio ช่วยให้จัดลำดับความสำคัญของตลาดได้ชัดเจน  
- สามารถใช้กำหนดกลยุทธ์แบบ **Diversification + Focused Investment**

ดังนั้น กราฟนี้สะท้อนว่า **“การบริหารตลาดนักท่องเที่ยวควรมองแบบพอร์ตโฟลิโอ เพื่อเลือกลงทุนในตลาดที่เหมาะสมที่สุดในแต่ละกลุ่ม”**

# One-Page Executive Dashboard

หลังจากวิเคราะห์ข้อมูลในแต่ละมิติแล้ว ขั้นตอนสำคัญคือการ “รวมทุก Insight เข้าด้วยกัน”  
เพื่อสร้างมุมมองแบบ **Executive Dashboard** ที่สามารถเข้าใจได้ภายในครั้งเดียว

Dashboard นี้ถูกออกแบบมาเพื่อสื่อสาร **Insight ที่สำคัญที่สุดในระดับกลยุทธ์**  
ช่วยให้ผู้บริหารสามารถ “มองภาพรวมและตัดสินใจได้ทันที” โดยไม่ต้องไล่ดูรายละเอียดทีละกราฟ

## เป้าหมายของ Dashboard
- เห็นภาพรวมของตลาดนักท่องเที่ยวในหน้าเดียว  
- เปรียบเทียบทั้ง **ขนาดตลาด (Size)** และ **การเติบโต (Growth)** ได้ทันที  
- เข้าใจ **Seasonality และพฤติกรรมการเดินทาง** อย่างรวดเร็ว  
- ระบุ **ตลาดหลัก ตลาดเกิดใหม่ และตลาดเสี่ยง** ได้ในมุมมองเดียว  

In [ ]:
# PREPARE DATA FOR PREMIUM DASHBOARD
kpi = con.sql("""
SELECT
    SUM(arr_2025_YTD) AS total_ytd_arrivals,
    AVG(pct_change_YTD) AS avg_pct_change,
    COUNT(*) AS total_countries
FROM tourism_country
""").df()

top10_ytd = con.sql("""
SELECT nationality, arr_2025_YTD, pct_change_YTD
FROM tourism_country
ORDER BY arr_2025_YTD DESC
LIMIT 10
""").df()

bubble_df = con.sql("""
SELECT nationality, arr_2025_YTD, pct_change_YTD
FROM tourism_country
WHERE arr_2025_YTD IS NOT NULL
  AND pct_change_YTD IS NOT NULL
""").df()

monthly_total = con.sql("""
SELECT month, month_num, SUM(arrivals_2025) AS total_arrivals
FROM tourism_long_2025
GROUP BY month, month_num
ORDER BY month_num
""").df()

monthly_country = con.sql("""
SELECT nationality, month, month_num, arrivals_2025
FROM tourism_long_2025
WHERE entity_type = 'Country'
""").df()

monthly_country = monthly_country.dropna(subset=["arrivals_2025"]).copy()

monthly_top10 = (
    monthly_country
    .sort_values(["month_num", "arrivals_2025"], ascending=[True, False])
    .groupby("month", group_keys=False)
    .head(10)
    .copy()
)

monthly_top10 = monthly_top10.sort_values(["month_num", "arrivals_2025"], ascending=[True, True])

In [ ]:
# ONE-PAGE DASHBOARD
fig = make_subplots(
    rows=2, cols=2,
    specs=[
        [{"type": "bar"}, {"type": "scatter"}],
        [{"type": "scatter"}, {"type": "bar"}]
    ],
    subplot_titles=(
        "Top 10 Tourist Markets (YTD 2025)",
        "Market Positioning: Size vs Growth",
        "Monthly Arrivals Trend (2025)",
        "Monthly Leaderboard Snapshot"
    ),
    horizontal_spacing=0.08,
    vertical_spacing=0.14
)

# Panel 1
fig.add_trace(
    go.Bar(
        x=top10_ytd["arr_2025_YTD"],
        y=top10_ytd["nationality"],
        orientation="h",
        marker=dict(
            color=top10_ytd["arr_2025_YTD"],
            colorscale=[
                [0.00, "#593222"],
                [0.35, "#8A3324"],
                [0.65, "#D36228"],
                [1.00, "#F5E5C0"]
            ],
            line=dict(color="rgba(255,255,255,0.15)", width=1),
            showscale=False
        ),
        text=[fmt_int(v) for v in top10_ytd["arr_2025_YTD"]],
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>YTD Arrivals: %{x:,.0f}<extra></extra>"
    ),
    row=1, col=1
)

# Panel 2
fig.add_trace(
    go.Scatter(
        x=bubble_df["arr_2025_YTD"],
        y=bubble_df["pct_change_YTD"],
        mode="markers",
        text=bubble_df["nationality"],
        marker=dict(
            size=np.clip(np.sqrt(bubble_df["arr_2025_YTD"]) / 18, 8, 48),
            color=bubble_df["pct_change_YTD"],
            colorscale="Turbo",
            line=dict(color="rgba(255,255,255,0.2)", width=1),
            showscale=True,
            colorbar=dict(
                title="YoY %",
                x=1.02,
                y=0.77,
                len=0.35,
                bgcolor=COLOR_PANEL,
                tickfont=dict(color=COLOR_TEXT),
                titlefont=dict(color=COLOR_TEXT)
            )
        ),
        hovertemplate="<b>%{text}</b><br>YTD Arrivals: %{x:,.0f}<br>YoY Growth: %{y:.1f}%<extra></extra>"
    ),
    row=1, col=2
)

x_med = bubble_df["arr_2025_YTD"].median()
y_med = bubble_df["pct_change_YTD"].median()
fig.add_vline(x=x_med, line_dash="dot", line_color="rgba(255,255,255,0.25)", row=1, col=2)
fig.add_hline(y=y_med, line_dash="dot", line_color="rgba(255,255,255,0.25)", row=1, col=2)

top10_ytd = top10_ytd.sort_values("arr_2025_YTD", ascending=False)
fig.update_yaxes(autorange="reversed", row=1, col=1)

# Panel 3
fig.add_trace(
    go.Scatter(
        x=monthly_total["month"],
        y=monthly_total["total_arrivals"],
        mode="lines+markers+text",
        line=dict(color=COLOR_ORANGE, width=4),
        marker=dict(size=9, color=COLOR_GOLD, line=dict(color="#FFF4D6", width=1.2)),
        text=[fmt_int(v) for v in monthly_total["total_arrivals"]],
        textposition="top center",
        hovertemplate="<b>%{x}</b><br>Total Arrivals: %{y:,.0f}<extra></extra>"
    ),
    row=2, col=1
)

# Panel 4
snapshot_month = "Dec"
snapshot = monthly_top10[monthly_top10["month"] == snapshot_month].sort_values("arrivals_2025", ascending=True)

fig.add_trace(
    go.Bar(
        x=snapshot["arrivals_2025"],
        y=snapshot["nationality"],
        orientation="h",
        marker=dict(
            color=snapshot["arrivals_2025"],
            colorscale=[
                [0.00, "#1F2937"],
                [0.25, "#4E6E58"],
                [0.60, "#0F766E"],
                [1.00, "#F5E5C0"]
            ],
            line=dict(color="rgba(255,255,255,0.15)", width=1),
            showscale=False
        ),
        text=[fmt_int(v) for v in snapshot["arrivals_2025"]],
        textposition="outside",
        hovertemplate=f"<b>%{{y}}</b><br>Month: {snapshot_month}<br>Arrivals: %{{x:,.0f}}<extra></extra>"
    ),
    row=2, col=2
)

# KPI annotations
total_ytd = kpi.loc[0, "total_ytd_arrivals"]
avg_growth = kpi.loc[0, "avg_pct_change"]
total_countries = kpi.loc[0, "total_countries"]

peak_row = monthly_total.loc[monthly_total["total_arrivals"].idxmax()]
low_row  = monthly_total.loc[monthly_total["total_arrivals"].idxmin()]

fig.add_annotation(
    x=0.03, y=1.15, xref="paper", yref="paper",
    text=f"<b>Total YTD Arrivals</b><br><span style='font-size:24px'>{fmt_int(total_ytd)}</span>",
    showarrow=False, align="left",
    font=dict(color=COLOR_TEXT, size=14),
    bgcolor="#7C2D12", bordercolor="rgba(255,255,255,0.12)", borderwidth=1, borderpad=10
)

fig.add_annotation(
    x=0.25, y=1.15, xref="paper", yref="paper",
    text=f"<b>Average YoY Growth</b><br><span style='font-size:24px'>{fmt_pct(avg_growth)}</span>",
    showarrow=False, align="left",
    font=dict(color=COLOR_TEXT, size=14),
    bgcolor="#7F1D1D", bordercolor="rgba(255,255,255,0.12)", borderwidth=1, borderpad=10
)

fig.add_annotation(
    x=0.55, y=1.15, xref="paper", yref="paper",
    text=f"<b>Countries in Dataset</b><br><span style='font-size:24px'>{fmt_int(total_countries)}</span>",
    showarrow=False, align="left",
    font=dict(color=COLOR_TEXT, size=14),
    bgcolor="#14532D", bordercolor="rgba(255,255,255,0.12)", borderwidth=1, borderpad=10
)

fig.add_annotation(
    x=0.85, y=1.15, xref="paper", yref="paper",
    text=(
        f"<b>Peak / Lowest Month</b><br>"
        f"<span style='font-size:18px'>{peak_row['month']} / {low_row['month']}</span>"
    ),
    showarrow=False, align="left",
    font=dict(color=COLOR_TEXT, size=14),
    bgcolor="#1E3A8A", bordercolor="rgba(255,255,255,0.12)", borderwidth=1, borderpad=10
)
fig.update_layout(
    title=dict(
        text="<b>Thailand Tourism Dashboard 2025</b><br><span style='font-size:13px;color:#C9B79C'>Executive view of tourist arrivals, growth, seasonality, and market leadership</span>",
        x=0.5,
        xanchor="center",
        y=0.97
    ),
    height=920,
    width=1500,
    paper_bgcolor=COLOR_BG,
    plot_bgcolor=COLOR_PANEL,
    font=dict(color=COLOR_TEXT, family="Georgia"),
    margin=dict(t=230, l=70, r=70, b=60),  # เพิ่ม top margin
    showlegend=False
)
fig.update_layout(
    title=dict(
        text="<b>Thailand Tourism Dashboard 2025</b><br><span style='font-size:13px;color:#C9B79C'>Executive view of tourist arrivals, growth, seasonality, and market leadership</span>",
        x=0.5,
        xanchor="center"
    ),
    height=920,
    width=1500,
    paper_bgcolor=COLOR_BG,
    plot_bgcolor=COLOR_PANEL,
    font=dict(color=COLOR_TEXT, family="Georgia"),
    margin=dict(t=170, l=70, r=70, b=60),
    showlegend=False
)

fig.update_xaxes(showgrid=True, gridcolor=COLOR_GRID, zeroline=False)
fig.update_yaxes(showgrid=False, zeroline=False)

fig.update_xaxes(title_text="YTD Arrivals", row=1, col=1)
fig.update_yaxes(title_text="", row=1, col=1)

fig.update_xaxes(title_text="YTD Arrivals", row=1, col=2)
fig.update_yaxes(title_text="YoY Growth (%)", row=1, col=2)

fig.update_xaxes(title_text="Month", categoryorder="array", categoryarray=MONTH_ORDER, row=2, col=1)
fig.update_yaxes(title_text="Total Arrivals", row=2, col=1)

fig.update_xaxes(title_text="Monthly Arrivals", row=2, col=2)
fig.update_yaxes(title_text="", row=2, col=2)

fig.show()

### อธิบายผลการวิเคราะห์ Dashboard

Dashboard หน้าเดียวนี้สรุปภาพรวมของการท่องเที่ยวไทยในปี 2025 ไว้อย่างครบถ้วน  
โดยเชื่อมโยงทั้ง “ขนาดตลาด”, “การเติบโต”, “seasonality” และ “ผู้นำรายเดือน” ไว้ในมุมมองเดียว

#### สิ่งที่เห็นจาก Dashboard
- **ซ้ายบน:** ตลาดหลักของไทยยังคงกระจุกตัวอยู่ในไม่กี่ประเทศ โดยเฉพาะ **Malaysia และ China**  
- **ขวาบน:** เมื่อนำ “ขนาดตลาด” และ “การเติบโต” มาพิจารณาร่วมกัน จะเห็นความแตกต่างของแต่ละตลาดอย่างชัดเจน  
- **ซ้ายล่าง:** จำนวนนักท่องเที่ยวมี **seasonality ชัดเจน** โดยต้นปีสูง กลางปีอ่อนตัว และปลายปีฟื้นกลับขึ้นมา  
- **ขวาล่าง:** ใน snapshot รายเดือน จะเห็นว่าประเทศผู้นำในเดือนล่าสุด (Dec'2025) ยังคงเป็นตลาดหลักเดิมเป็นส่วนใหญ่

#### การตีความเชิงกลยุทธ์
- ไทยยังพึ่งพาตลาดหลักในเอเชียค่อนข้างมาก  
- การท่องเที่ยวไทยมีจังหวะ demand ตามฤดูกาลที่ชัดเจน  
- บางตลาดมีบทบาทสำคัญในเชิง “ขนาด” ขณะที่บางตลาดโดดเด่นในเชิง “การเติบโต”  
- การวางกลยุทธ์จึงควรมองทั้งภาพรวมและรายตลาดควบคู่กัน

ดังนั้น Dashboard นี้ช่วยยืนยันว่า **“การท่องเที่ยวไทยในปี 2025 ยังขับเคลื่อนโดยตลาดหลักไม่กี่ประเทศ มี seasonality ชัดเจน และควรมองการบริหารตลาดแบบเชิงพอร์ตโฟลิโอ”**

# Animated Monthly Leaderboard

เพื่อเพิ่มพลังในการเล่าเรื่อง เราจะสร้างกราฟแบบ animation  
ให้เห็นว่าในแต่ละเดือน ประเทศใดเป็นผู้นำ และอันดับเปลี่ยนแปลงไปอย่างไรตลอดปี 2025

In [ ]:
# ANIMATED MONTHLY LEADERBOARD
anim_df = monthly_top10.copy()
anim_df["month"] = pd.Categorical(anim_df["month"], categories=MONTH_ORDER, ordered=True)
anim_df = anim_df.sort_values(["month", "arrivals_2025"], ascending=[True, True])

fig_anim = px.bar(
    anim_df,
    x="arrivals_2025",
    y="nationality",
    orientation="h",
    animation_frame="month",
    animation_group="nationality",
    color="arrivals_2025",
    text="arrivals_2025",
    range_x=[0, anim_df["arrivals_2025"].max() * 1.18],
    color_continuous_scale=[
        "#442F23", "#593222", "#8A3324", "#B3342C", "#D36228", "#F2882F", "#F5E5C0"
    ],
    title="Top 10 Tourist Markets by Month (Animated)"
)

fig_anim.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    marker_line_color="rgba(255,255,255,0.15)",
    marker_line_width=1.1,
    hovertemplate="<b>%{y}</b><br>Arrivals: %{x:,.0f}<extra></extra>"
)

fig_anim.update_layout(
    height=720,
    width=1250,
    paper_bgcolor=COLOR_BG,
    plot_bgcolor=COLOR_PANEL,
    font=dict(color=COLOR_TEXT, family="Georgia"),
    title=dict(
        x=0.5,
        text="<b>Thailand Tourism Monthly Race 2025</b><br><span style='font-size:13px;color:#C9B79C'>Animated ranking of top tourist markets by month</span>"
    ),
    margin=dict(t=110, l=80, r=80, b=70),
    coloraxis_showscale=False
)

fig_anim.update_xaxes(
    title="Monthly Arrivals",
    showgrid=True,
    gridcolor=COLOR_GRID,
    zeroline=False
)

fig_anim.update_yaxes(
    title="",
    categoryorder="total ascending"
)

# Speed control
if fig_anim.layout.updatemenus:
    fig_anim.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 900
    fig_anim.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 500

if fig_anim.layout.sliders:
    fig_anim.layout.sliders[0]["currentvalue"]["prefix"] = "Month: "
    fig_anim.layout.sliders[0]["currentvalue"]["font"] = dict(color=COLOR_TEXT, size=16)

fig_anim.show()

### อธิบายผลการวิเคราะห์ Animation

กราฟ Animation นี้แสดงการเปลี่ยนแปลงอันดับของประเทศนักท่องเที่ยวแบบรายเดือนตลอดปี 2025  
โดยเมื่อกด Play จะเห็น “การแข่งขันของตลาดต้นทาง” เปลี่ยนแปลงอย่างต่อเนื่องในแต่ละช่วงเวลา

#### ข้อสังเกตสำคัญ
- ประเทศหลักอย่าง **China และ Malaysia** ยังคงอยู่ในอันดับต้นอย่างต่อเนื่อง  
- บางประเทศมีการ “ไต่ขึ้น” หรือ “ตกลง” ของอันดับในบางช่วงเดือน  
- การเปลี่ยนแปลงอันดับสะท้อนถึง seasonality และพฤติกรรมการเดินทางที่แตกต่างกัน

#### การตีความ
- ตลาดนักท่องเที่ยวไม่ได้มีโครงสร้างคงที่ แต่มี **dynamic movement ตลอดทั้งปี**  
- การขึ้นลงของอันดับแสดงถึงช่วงเวลาที่แต่ละประเทศมี demand สูงหรือต่ำ  
- ช่วยให้เข้าใจ “momentum ของตลาด” ได้ชัดเจนกว่ากราฟนิ่ง

# Final Conclusion

จากคำถามสำคัญทั้ง 3 ข้อ ได้แก่  
“โครงสร้างตลาดเป็นอย่างไร”, “นักท่องเที่ยวมาเมื่อไหร่”, และ “ตลาดกำลังเปลี่ยนไปทางไหน”  
การวิเคราะห์ข้อมูลปี 2025 ช่วยให้เห็นภาพของการท่องเที่ยวไทยได้อย่างชัดเจนในมุมมองเชิงกลยุทธ์

### 1) โครงสร้างตลาด (Market Structure)
ตลาดนักท่องเที่ยวของไทยยังคง **กระจุกตัวอยู่ในไม่กี่ประเทศหลัก**  
โดยเฉพาะ **Malaysia และ China** ซึ่งมีบทบาทสูงอย่างโดดเด่น  

➡️ สะท้อนว่าไทยยังเป็น **Regional-driven tourism economy** ที่พึ่งพาตลาดเอเชียเป็นหลัก

---

### 2) Seasonality ของการท่องเที่ยว
การเดินทางมี **seasonality ชัดเจน**
- **Peak:** ต้นปี (Jan)  
- **Low:** กลางปี – Sep  
- **Recovery:** ปลายปี  

และที่สำคัญ แต่ละประเทศมี “จังหวะการเดินทางต่างกัน”  
➡️ ทำให้ demand ของไทยเป็น **multi-seasonality structure** ไม่ได้ขึ้นลงแบบเดียวกันทั้งหมด

---

### 3) การเปลี่ยนแปลงของตลาด (Market Shift)
โครงสร้างตลาดกำลังเริ่มเปลี่ยนแปลง
- มี **Emerging markets** ที่เติบโตเร็ว  
- ตลาดใหญ่บางประเทศเริ่ม **ชะลอหรือหดตัว**  
- ตลาดที่มีทั้ง “ขนาดใหญ่ + เติบโต” (เช่น India) เริ่มมีบทบาทมากขึ้น  

➡️ สะท้อนว่าไทยกำลังเปลี่ยนจากการพึ่งพาตลาดเดิม  
ไปสู่การบริหารแบบ **diversified tourism portfolio**

---

###Key Takeaway

การท่องเที่ยวไทยในปี 2025 ไม่ได้เป็นเพียงตลาดเดียว  
แต่เป็นระบบที่ขับเคลื่อนด้วย “ตลาดหลัก + จังหวะเวลา + การเปลี่ยนแปลงของ demand”

ความสำเร็จในอนาคตจะขึ้นอยู่กับความสามารถในการ  
**บริหารโครงสร้างตลาดให้สมดุล และใช้ประโยชน์จากความหลากหลายของ demand ได้อย่างมีประสิทธิภาพ**




# Strategic Recommendations

จาก Insight ทั้งหมด สามารถกำหนดกลยุทธ์ได้ดังนี้:

## 1) Diversification Strategy (ลดความเสี่ยง)
- ลดการพึ่งพา **China / Malaysia**
- ขยายไปยัง **Emerging markets ที่โตเร็ว**
- กระจาย demand ให้สมดุลมากขึ้น

---

## 2) Market Prioritization (โฟกัสตลาดสำคัญ)
แบ่งกลุ่มตลาดเป็น 4 กลยุทธ์:

- 🔥 **Strategic Priority (ใหญ่ + โต)** → ลงทุนเต็มที่ (เช่น India)
- 🏢 **Core Markets (ใหญ่ แต่โตช้า)** → รักษาฐาน (เช่น China)
- 🌱 **Emerging Markets (เล็ก แต่โตเร็ว)** → ทดลอง & scale
- 👀 **Monitor Markets** → ติดตาม ไม่เร่งลงทุน

---

## 3) Seasonality Optimization
- ใช้ **country-specific campaign**
- ดึงนักท่องเที่ยวต่างประเทศในช่วง low season
- ใช้ **dynamic pricing + promotion** ปรับตามช่วงเวลา

---

## 4) Revenue Strategy (ไม่ใช่แค่จำนวนคน)
- โฟกัสตลาดที่มี **high spending per capita**
- ดึง long-haul markets (EU / US) เพิ่ม
- เพิ่มคุณภาพมากกว่าปริมาณ

---

## 5) Data-Driven Tourism Management
- ใช้ dashboard แบบนี้ในการ monitor real-time
- ทำ **early warning system** สำหรับตลาดที่หดตัว
- ใช้ analytics วางแผนเชิงกลยุทธ์ระยะยาว

---

# Final Insight

ประเทศไทยไม่ได้มี “ตลาดเดียว”  
แต่กำลังบริหาร “พอร์ตของตลาดนักท่องเที่ยว”

ผู้ชนะในอนาคตจะไม่ใช่ประเทศที่มีนักท่องเที่ยวมากที่สุด  
แต่คือประเทศที่ **บริหาร portfolio ของตลาดได้ดีที่สุด**

**ผู้จัดทำ : สุจิวรรณ กิจสุธรรม ID 6810422019**